# ✂️ Özellik Seçimi ve Modeli Yorumlama (Feature Selection & Explainable AI)

Gürültü yapan gereksiz sütunları silmek (Feature Selection) ve modelin nasıl karar verdiğini anlamak (SHAP, Permutation Importance) için Kaggle'da kullanılan yöntemler.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.inspection import permutation_importance
import shap

## 1. Ağaç Tabanlı Özellik Önemi (Tree-based Feature Importance)

In [ ]:
def plot_tree_importance(model, X_train, top_n=20):
    """
    Eğitilmiş bir Random Forest, XGBoost veya LightGBM modelinin varsayılan (Gini/Split) özelliğini çizer.
    """
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1][:top_n]
    
    plt.figure(figsize=(10, 8))
    plt.title(f"En Önemli {top_n} Özellik (Tree Importance)")
    sns.barplot(x=importances[indices], y=X_train.columns[indices], palette="viridis")
    plt.show()

# KULLANIM:
# plot_tree_importance(xgb_model, X_train)

## 2. Permutation Importance (Daha Güvenilir)
Özellik sütunlarını tek tek karıştırıp modelin hatasının ne kadar arttığına bakar. Eğer hata artıyorsa o özellik çok önemlidir.

In [ ]:
def calculate_permutation_importance(model, X_val, y_val, scoring='neg_mean_squared_error'):
    """
    Önemli: Permutation Importance'ı Test veya Validation seti üzerinde çalıştırmak (X_val, y_val)
    modelin ezberleme (overfitting) yapıp yapmadığını anlamak için çok daha etkilidir.
    """
    result = permutation_importance(model, X_val, y_val, n_repeats=10, random_state=42, scoring=scoring)
    
    sorted_idx = result.importances_mean.argsort()
    
    plt.figure(figsize=(10, 8))
    plt.boxplot(
        result.importances[sorted_idx].T,
        vert=False,
        labels=X_val.columns[sorted_idx]
    )
    plt.title("Permutation Importances (Validation Set)")
    plt.tight_layout()
    plt.show()
    
    return result

## 3. SHAP (SHapley Additive exPlanations) - Altın Standart
Bir özelliğin sadece ne kadar önemli olduğunu değil, hedefe HANGİ YÖNDE (pozitif mi negatif mi) etki ettiğini gösterir.

In [ ]:
def plot_shap_summary(model, X_train):
    """
    Ağaç modelleri (TreeExplainer) için genel SHAP özet grafiğini (Summary Plot) çizer.
    Kırmızı noktalar: O özelliğin yüksek değeri.
    Sağ taraf: Hedef değişkeni pozitif etkileyenler (Örn: Fiyatı artıranlar).
    """
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_train)
    
    # LightGBM Classifier gibi bazı modellerde shap_values list olarak dönebilir (Sınıflar için)
    if isinstance(shap_values, list):
        shap_values = shap_values[1] # Pozitif sınıfı çiz
        
    shap.summary_plot(shap_values, X_train)

# KULLANIM:
# plot_shap_summary(lgbm_model, X_train)